# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fksifat/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I’m using a small comparison set: logistic regression as the readable anchor, a decision tree as a simple nonlinear check, and a random forest as the strongest candidate. That fits this lane because the target is binary (`is_declining_label`), the baseline is a rank-based rule, and the question is whether a learned model can improve Precision@50 on the same held-out clients without using label-derived columns.

I will not use `trend_direction` or `trend_pct` as features because they define the label. The split stays client-holdout so pages from the same client do not leak across train and test.

In [6]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    data_candidate = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
    if data_candidate.exists():
        repo_root = candidate
        break

data_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
baseline_queue_path = repo_root / "data" / "processed" / "baseline_refresh_queue.csv"
model_results_path = repo_root / "outputs" / "model_results.json"

frame = pd.read_csv(data_path)
baseline_queue = pd.read_csv(baseline_queue_path)
reference_results = json.loads(model_results_path.read_text())

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]


def build_feature_matrix(input_frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    numeric_features = [column for column in MODEL_NUMERIC_FEATURES if column in input_frame.columns]
    categorical_features = [column for column in MODEL_CATEGORICAL_FEATURES if column in input_frame.columns]

    numeric_frame = input_frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)

    categorical_frame = input_frame[categorical_features].fillna("unknown").astype(str)
    encoded_frame = pd.get_dummies(
        categorical_frame,
        prefix=categorical_features,
        dummy_na=False,
        dtype=float,
    )

    feature_frame = pd.concat(
        [numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)],
        axis=1,
    )
    return feature_frame, list(feature_frame.columns)


def make_client_aware_split(
    input_frame: pd.DataFrame,
    target_series: pd.Series,
) -> tuple[np.ndarray, np.ndarray, str]:
    all_indices = np.arange(len(input_frame))
    client_series = input_frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()

    if len(unique_clients) >= 5:
        random_generator = np.random.default_rng(RANDOM_STATE)
        shuffled_clients = random_generator.permutation(unique_clients)
        test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
        test_clients = set(shuffled_clients[:test_client_count])
        test_mask = client_series.isin(test_clients).to_numpy()
        train_indices = all_indices[~test_mask]
        test_indices = all_indices[test_mask]

        if (
            len(train_indices) > 0
            and len(test_indices) > 0
            and target_series.iloc[train_indices].nunique() == 2
            and target_series.iloc[test_indices].nunique() == 2
        ):
            return train_indices, test_indices, "client_holdout"

    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=target_series,
    )
    return np.array(train_indices), np.array(test_indices), "stratified_row_holdout"


def precision_at_k(y_true: pd.Series, scores: np.ndarray, k: int) -> float:
    ranking = pd.DataFrame({"y": y_true.to_numpy(), "score": np.asarray(scores, dtype=float)})
    if ranking.empty:
        return 0.0
    top = ranking.sort_values("score", ascending=False).head(min(k, len(ranking)))
    return float(top["y"].mean()) if len(top) else 0.0


def build_models() -> dict[str, object]:
    return {
        "logistic_regression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=1000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "decision_tree": DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=5,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        ),
        "random_forest": RandomForestClassifier(
            class_weight="balanced_subsample",
            max_depth=10,
            min_samples_leaf=25,
            n_estimators=200,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    }


def predict_probability(model: object, feature_frame: pd.DataFrame) -> np.ndarray:
    return np.asarray(model.predict_proba(feature_frame)[:, 1], dtype=float)


def metric_payload(target_series: pd.Series, probability_scores: np.ndarray, prefix: str = "") -> dict[str, float]:
    binary_predictions = (probability_scores >= 0.5).astype(int)
    payload = {
        f"{prefix}accuracy": float(accuracy_score(target_series, binary_predictions)),
        f"{prefix}precision": float(precision_score(target_series, binary_predictions, zero_division=0)),
        f"{prefix}recall": float(recall_score(target_series, binary_predictions, zero_division=0)),
        f"{prefix}f1": float(f1_score(target_series, binary_predictions, zero_division=0)),
        f"{prefix}precision_at_20": precision_at_k(target_series, probability_scores, 20),
        f"{prefix}precision_at_50": precision_at_k(target_series, probability_scores, 50),
        f"{prefix}precision_at_100": precision_at_k(target_series, probability_scores, 100),
    }
    payload[f"{prefix}roc_auc"] = float(roc_auc_score(target_series, probability_scores))
    payload[f"{prefix}average_precision"] = float(average_precision_score(target_series, probability_scores))
    return payload


feature_frame, feature_columns = build_feature_matrix(frame)
target_series = frame["is_declining_label"].astype(int)
train_indices, test_indices, split_strategy = make_client_aware_split(frame, target_series)
train_features = feature_frame.iloc[train_indices]
test_features = feature_frame.iloc[test_indices]
train_target = target_series.iloc[train_indices]
test_target = target_series.iloc[test_indices]

assert "trend_direction" not in feature_columns
assert "trend_pct" not in feature_columns
assert split_strategy == "client_holdout"

print(f"Loaded {len(frame):,} rows with {len(feature_columns):,} model features.")
print(f"Target rate: {target_series.mean():.3f} declining.")
print(f"Train rows: {len(train_indices):,} | Test rows: {len(test_indices):,}")
print(f"Held-out clients: {frame.iloc[test_indices]['client_id'].nunique():,}")

Loaded 30,000 rows with 52 model features.
Target rate: 0.542 declining.
Train rows: 27,675 | Test rows: 2,325
Held-out clients: 6


## 2. Split design

I’m using a client-holdout split, not a row-level split. That is the honest design here because the same client can produce many pages with shared operating patterns, and letting pages from one client appear in both train and test would leak client-specific signal into validation. The notebook reproduces the same split strategy used by the training script, so the comparison stays aligned with the Week-4 baseline.

In [7]:
split_summary = pd.DataFrame(
    {
        "set": ["train", "test"],
        "rows": [len(train_indices), len(test_indices)],
        "clients": [frame.iloc[train_indices]["client_id"].nunique(), frame.iloc[test_indices]["client_id"].nunique()],
        "declining_rate": [train_target.mean(), test_target.mean()],
    }
)

reference_split = reference_results["split_strategy"]
reference_test_rows = reference_results["test_rows"]
reference_train_rows = reference_results["train_rows"]

print(f"Split strategy from saved results: {reference_split}")
print(f"Rows match saved results: train={len(train_indices) == reference_train_rows}, test={len(test_indices) == reference_test_rows}")
print("\nSplit summary:")
print(split_summary.to_string(index=False, formatters={"declining_rate": "{:.3f}".format}))

baseline_scores = baseline_queue.set_index("content_id")["baseline_refresh_score"]
baseline_test_scores = frame.iloc[test_indices]["content_id"].map(baseline_scores).fillna(0).to_numpy()
baseline_metrics = metric_payload(test_target, baseline_test_scores, prefix="baseline_")

print("\nBaseline check against saved metrics:")
print(f"baseline_precision_at_50 = {baseline_metrics['baseline_precision_at_50']:.3f} (saved: {reference_results['baseline']['baseline_precision_at_50']:.3f})")
print(f"baseline_roc_auc = {baseline_metrics['baseline_roc_auc']:.3f} (saved: {reference_results['baseline']['baseline_roc_auc']:.3f})")
assert np.isclose(baseline_metrics['baseline_precision_at_50'], reference_results['baseline']['baseline_precision_at_50'])
assert np.isclose(baseline_metrics['baseline_roc_auc'], reference_results['baseline']['baseline_roc_auc'])

Split strategy from saved results: client_holdout
Rows match saved results: train=True, test=True

Split summary:
  set  rows  clients declining_rate
train 27675       26          0.555
 test  2325        6          0.391

Baseline check against saved metrics:
baseline_precision_at_50 = 0.240 (saved: 0.240)
baseline_roc_auc = 0.627 (saved: 0.627)


## 3. Train + compare vs my baseline

I train the three candidate models on the same encoded feature matrix and compare them on the same held-out clients and the same ranking metric, Precision@50. I keep the baseline rule in the same table so the result is judged on the same split, not on a separate convenience metric.

In [8]:
models = build_models()
model_metrics: dict[str, dict[str, float]] = {}
trained_models: dict[str, object] = {}

for model_name, model in models.items():
    model.fit(train_features, train_target)
    trained_models[model_name] = model
    test_probabilities = predict_probability(model, test_features)
    model_metrics[model_name] = metric_payload(test_target, test_probabilities)

comparison_rows = []
for model_name, metrics in model_metrics.items():
    comparison_rows.append(
        {
            "model": model_name,
            "roc_auc": metrics["roc_auc"],
            "average_precision": metrics["average_precision"],
            "precision_at_50": metrics["precision_at_50"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        }
    )
comparison_rows.append(
    {
        "model": "baseline_rules",
        "roc_auc": baseline_metrics["baseline_roc_auc"],
        "average_precision": baseline_metrics["baseline_average_precision"],
        "precision_at_50": baseline_metrics["baseline_precision_at_50"],
        "recall": np.nan,
        "f1": np.nan,
    }
)
comparison_table = pd.DataFrame(comparison_rows).sort_values(
    by=["precision_at_50", "average_precision", "roc_auc"],
    ascending=False,
).reset_index(drop=True)

best_model_name = comparison_table.loc[0, "model"]
best_model = trained_models[best_model_name]
best_test_probabilities = predict_probability(best_model, test_features)
best_precision_at_50 = comparison_table.loc[0, "precision_at_50"]
baseline_precision_at_50 = baseline_metrics["baseline_precision_at_50"]
precision_lift = best_precision_at_50 - baseline_precision_at_50

print("Model-vs-baseline table:")
print(comparison_table.to_string(index=False, formatters={
    "roc_auc": "{:.3f}".format,
    "average_precision": "{:.3f}".format,
    "precision_at_50": "{:.3f}".format,
    "recall": lambda value: "-" if pd.isna(value) else f"{value:.3f}",
    "f1": lambda value: "-" if pd.isna(value) else f"{value:.3f}",
}))
print(f"\nSelected model: {best_model_name}")
print(f"Precision@50 lift vs baseline: {precision_lift:+.3f}")

Model-vs-baseline table:
              model roc_auc average_precision precision_at_50 recall    f1
      random_forest   0.750             0.618           0.740  0.744 0.640
      decision_tree   0.742             0.575           0.660  0.716 0.634
logistic_regression   0.700             0.522           0.400  0.567 0.566
     baseline_rules   0.627             0.468           0.240    NaN   NaN

Selected model: random_forest
Precision@50 lift vs baseline: +0.500


## 4. Errors and interpretation

I’m checking the model’s mistakes before trusting the score. The two questions are: which examples did it call decline with high confidence but the label did not agree, and which true declines did it miss? I also inspect the strongest feature importances to make sure the model is learning plausible operating signals instead of something suspicious.

In [9]:
prediction_frame = frame.iloc[test_indices][
    [
        "content_id",
        "client_id",
        "content_type",
        "impression_tier",
        "position_tier",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "is_declining_label",
    ]
].copy()
prediction_frame["predicted_probability"] = best_test_probabilities
prediction_frame["predicted_label"] = (prediction_frame["predicted_probability"] >= 0.5).astype(int)
prediction_frame["error_type"] = np.select(
    [
        (prediction_frame["predicted_label"] == 1) & (prediction_frame["is_declining_label"] == 0),
        (prediction_frame["predicted_label"] == 0) & (prediction_frame["is_declining_label"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

feature_importance_frame = None
if best_model_name == "random_forest":
    importance_values = best_model.feature_importances_
elif best_model_name == "decision_tree":
    importance_values = best_model.feature_importances_
elif best_model_name == "logistic_regression":
    importance_values = np.abs(best_model.named_steps["model"].coef_[0])
else:
    importance_values = np.zeros(len(feature_columns), dtype=float)

feature_importance_frame = (
    pd.DataFrame({"feature": feature_columns, "importance": importance_values})
    .sort_values("importance", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

false_positives = prediction_frame[prediction_frame["error_type"] == "false_positive"].sort_values(
    "predicted_probability", ascending=False
)
false_negatives = prediction_frame[prediction_frame["error_type"] == "false_negative"].sort_values(
    "predicted_probability", ascending=True
)

print(f"Best model: {best_model_name}")
print("\nTop feature importances:")
print(feature_importance_frame.to_string(index=False, formatters={"importance": "{:.4f}".format}))

print("\nThree highest-confidence false positives:")
if len(false_positives) == 0:
    print("None in the held-out set.")
else:
    print(
        false_positives.head(3)[
            [
                "content_id",
                "content_type",
                "impression_tier",
                "position_tier",
                "content_age_days",
                "days_since_last_update",
                "ctr",
                "avg_position",
                "predicted_probability",
            ]
        ].to_string(index=False, formatters={"predicted_probability": "{:.3f}".format, "ctr": "{:.2f}".format, "avg_position": "{:.1f}".format})
    )

print("\nThree highest-confidence false negatives:")
if len(false_negatives) == 0:
    print("None in the held-out set.")
else:
    print(
        false_negatives.head(3)[
            [
                "content_id",
                "content_type",
                "impression_tier",
                "position_tier",
                "content_age_days",
                "days_since_last_update",
                "ctr",
                "avg_position",
                "predicted_probability",
            ]
        ].to_string(index=False, formatters={"predicted_probability": "{:.3f}".format, "ctr": "{:.2f}".format, "avg_position": "{:.1f}".format})
    )

error_slice = (
    prediction_frame.groupby(["content_type", "position_tier"], dropna=False)["error_type"]
    .value_counts()
    .rename("count")
    .reset_index()
)
print("\nError mix by content type and position tier (top 10 rows):")
print(error_slice.head(10).to_string(index=False))

Best model: random_forest

Top feature importances:
              feature importance
days_with_impressions     0.1350
  log_impressions_90d     0.1294
         avg_position     0.1092
     content_age_days     0.0920
           char_count     0.0387
        age_tier_365+     0.0368
       log_clicks_90d     0.0366
           word_count     0.0354
                  ctr     0.0352
          scroll_rate     0.0339

Three highest-confidence false positives:
          content_id    content_type impression_tier position_tier  content_age_days  days_since_last_update  ctr avg_position predicted_probability
content_d2dffcc697a4 keyword article            good      striking               144                      20 0.20         14.1                 0.737
content_00603b0349b4 keyword article        moderate      page_3_5               125                      20 0.09         25.6                 0.735
content_331182ca4cae keyword article            good      page_3_5               134           

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

I verified the notebook by running every code cell in order in this session.